# 3장 2강: 규제 — L1(Lasso), L2(Ridge), ElasticNet

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 Ridge, Lasso, ElasticNet 규제가 회귀계수와 일반화 성능에 어떤 영향을 주는지 확인합니다.

- 규제가 계수의 크기를 줄이는 이유를 이해합니다.
- `alpha` 변화에 따라 Ridge의 계수 크기와 Test 성능이 어떻게 달라지는지 확인합니다.
- Lasso가 일부 계수를 정확히 0으로 만드는지 확인합니다.
- ElasticNet이 L1과 L2를 함께 사용하는 모델이라는 점을 확인합니다.
- Ridge / Lasso / ElasticNet의 계수와 Test 성능을 비교합니다.

> 이 실습에서는 3장 2강 교안에서 다룬 규제, alpha, Lasso, Ridge, ElasticNet, StandardScaler, R² 비교만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- numpy
- matplotlib
- scikit-learn
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

규제에 따른 계수 변화를 비교하기 위해 결측치가 없는 숫자형 특성 중 다음 12개를 사용합니다.

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

규제는 계수의 크기를 기준으로 벌점을 주기 때문에 특성의 스케일을 맞추는 과정이 중요합니다.

이번 실습에서는 다음 순서로 데이터를 준비합니다.

1. Ames Housing 데이터 불러오기
2. 사용할 특성과 `SalePrice` 선택
3. Train / Test 분리
4. `StandardScaler`를 이용한 특성 표준화

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train 크기:", X_train_scaled.shape)
print("Test 크기:", X_test_scaled.shape)

Train 크기: (1168, 12)
Test 크기: (292, 12)


---

# 필수 1. Ridge와 alpha에 따른 계수 축소

## 배경

Ridge는 **L2 규제**를 사용합니다.

Ridge는 계수의 제곱합에 벌점을 주기 때문에 큰 계수를 더 강하게 억제하고, 모든 계수를 0에 가깝게 줄이는 특징이 있습니다.

## 문제 1. 여러 alpha 값에서 Ridge를 비교하세요.

### 요구사항

다음 alpha 값을 사용합니다.

```python
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
```

각 alpha에 대해 다음을 수행합니다.

1. `Ridge(alpha=alpha)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test 데이터를 예측합니다.
4. Test R²를 계산합니다.
5. `abs(model.coef_).sum()`으로 전체 계수 크기를 확인합니다.
6. 결과를 DataFrame으로 정리합니다.

### 확인 포인트

- alpha가 커질수록 규제 강도가 강해지는가?
- alpha가 커질수록 전체 계수 크기가 작아지는가?
- alpha가 너무 커지면 Test 성능이 낮아질 수 있는가?

In [12]:
# TODO
# 여러 alpha 값에 대해 Ridge의 Test R²와 계수 크기를 비교하세요.
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
result_r = []

for alpha in alpha_values:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)

    r2 = r2_score(y_test, pred)
    coef_sum = abs(model.coef_).sum()

    result_r.append([alpha, r2, coef_sum])

r_df = pd.DataFrame(result_r, columns=['alpha', 'test_r2', 'coef_sum'])
display(r_df)
print(result_r)

,alpha,test_r2,coef_sum
0,0.001,0.799357,102172.590850
1,0.010,0.799357,102172.463666
2,0.100,0.799355,102171.096834
3,1.000,0.799331,102149.831747
4,10.000,0.799056,101725.431454
5,100.000,0.795593,97390.120153


[[0.001, 0.7993573146096331, np.float64(102172.59084991204)], [0.01, 0.7993570831774636, np.float64(102172.46366619438)], [0.1, 0.7993547579607211, np.float64(102171.09683435432)], [1, 0.7993306096548939, np.float64(102149.8317465549)], [10, 0.7990555984835118, np.float64(101725.43145365473)], [100, 0.7955931222724344, np.float64(97390.12015341205)]]


### Q1. alpha가 커질수록 Ridge 계수는 일반적으로 어떻게 변하나요?

**답변:**  alpha가 커질수록 규제가 강해져서 계수가 전체적으로 0에 가깝게 줄어든다. 실제로 계수 크기 합이 alpha 0.001일 때 약 102,173에서 alpha 100일 때 약 97,390으로 줄었다.

### Q2. alpha가 너무 크면 어떤 문제가 발생할 수 있나요?

**답변:**  알파값이 크면 규제를 강하게 한다는 것이고 즉, 모델을 더 단순화 시킨다는 것이다. 모델을 적정값보다 더 단순화하면 과소적합이 발생할 수 있다.

---

# 필수 2. Lasso와 ElasticNet의 계수 변화

## 문제 1. Lasso에서 0이 되는 계수를 확인하세요.

### 배경

Lasso는 **L1 규제**를 사용하며, 중요하지 않은 특성의 계수를 정확히 0으로 만들 수 있습니다.

이번에는 규제 강도를 충분히 높여 Lasso의 변수 선택 효과를 직접 확인합니다.

### 요구사항

1. `Lasso(alpha=1000, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. 각 특성의 계수를 DataFrame으로 만듭니다.
4. 계수가 0인 특성의 개수를 계산합니다.
5. Test R²를 계산합니다.

### 확인 포인트

- 일부 계수가 정확히 0이 되는가?
- Ridge와 달리 Lasso가 변수 선택 효과를 갖는 이유를 이해했는가?

In [8]:
# TODO
# Lasso(alpha=1000)를 학습하고 계수와 0 계수 개수를 확인하세요.
lasso = Lasso(alpha=1000, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

lasso_coef = pd.DataFrame({
    'feature': features,
    'coef': lasso.coef_
})
display(lasso_coef)

zero_count = (lasso.coef_ == 0).sum()
lasso_r2 = r2_score(y_test, lasso.predict(X_test_scaled))
print(f'0인 계수 개수: {zero_count}개')
print(f'0인 특성: {list(lasso_coef[lasso_coef['coef'] == 0]['feature'])}')
print(f'Lasso Test R2: {lasso_r2}')

,feature,coef
0,OverallQual,25520.693859
1,GrLivArea,21419.631042
2,GarageCars,6820.209072
3,GarageArea,4370.090292
4,TotalBsmtSF,7305.573442
5,1stFlrSF,3521.096689
6,2ndFlrSF,0.000000
7,FullBath,-0.000000
8,TotRmsAbvGrd,83.200163
9,YearBuilt,6966.923170


0인 계수 개수: 2개
0인 특성: ['2ndFlrSF', 'FullBath']
Lasso Test R2: 0.7960182963880562


### Q3. Lasso에서 계수가 0이 된다는 것은 어떤 의미인가요?

**답변:**   해당 특성이 예측에 전혀 사용되지 않는다는 뜻이고, 사실상 모델에서 빠진 것과 같다. 이번에는 2ndFlrSF와 FullBath가 0이 됐다. 2ndFlrSF는 GrLivArea(생활 면적)에 이미 포함된 정보라서 Lasso가 중복된 특성을 빼고 GrLivArea 쪽에 계수를 몰아준 것 같다.

## 문제 2. ElasticNet을 학습하고 L1/L2 혼합을 확인하세요.

### 배경

ElasticNet은 L1과 L2 규제를 함께 사용합니다.

`l1_ratio`는 두 규제의 비율을 조절합니다.

- `l1_ratio`가 1에 가까움 → Lasso 성격이 강함
- `l1_ratio`가 0에 가까움 → Ridge 성격이 강함

### 요구사항

1. `ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test R²를 계산합니다.
4. 각 특성의 계수를 확인합니다.

### 확인 포인트

- `alpha`와 `l1_ratio`의 역할을 구분할 수 있는가?
- ElasticNet이 Lasso와 Ridge의 성격을 함께 가진다는 점을 설명할 수 있는가?

In [10]:
# TODO
# ElasticNet(alpha=0.1, l1_ratio=0.5)을 학습하세요.
enet = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
enet.fit(X_train_scaled, y_train)

enet_r2 = r2_score(y_test, enet.predict(X_test_scaled))
print(f'ElasticNet Test R2: {enet_r2}')

enet_coef = pd.DataFrame({
    'feature': features,
    'coef': enet.coef_
})
display(enet_coef)


ElasticNet Test R2: 0.7972913023796784


,feature,coef
0,OverallQual,23514.629015
1,GrLivArea,12670.768121
2,GarageCars,6785.127803
3,GarageArea,5137.437024
4,TotalBsmtSF,8016.089858
5,1stFlrSF,9237.815373
6,2ndFlrSF,6873.818086
7,FullBath,-1895.877032
8,TotRmsAbvGrd,2541.484569
9,YearBuilt,8072.285509


### Q4. `alpha`와 `l1_ratio`는 각각 무엇을 조절하나요?

**답변:**  
- alpha: 전체 규제의 강도를 조절한다. 클수록 계수를 더 많이 줄인다.
- l1_ratio: 규제 중에서 L1과 L2의 비율을 조절한다. 1에 가까우면 Lasso처럼 계수를 0으로 만들기 쉽고, 0에 가까우면 Ridge처럼 계수를 전체적으로 줄인다.

---

# 과제 1. 다중공선성 진단과 Ridge / Lasso / ElasticNet 비교

## 배경

같은 Train/Test 데이터에서 세 규제 모델을 비교합니다. 성취도 평가항목에 맞춰, 비교 전에 **다중공선성의 계수 해석 영향과 VIF 진단·처치 선택**을 추가합니다.

다중공선성은 입력 특성들이 서로 강한 선형 관계를 갖는 상황입니다. VIF는 한 특성을 나머지 특성으로 설명할 때 구한 결정계수 R²를 이용해 `1 / (1 - R²)`로 계산합니다. 이때 예측 대상은 `SalePrice`가 아니라 **진단할 입력 특성 하나**이며, 보조 회귀에는 절편을 포함합니다.

## 요구사항

### A. 다중공선성 진단과 처치 방향 선택

1. **Train의 입력 특성만** 이용해 특성별 VIF를 계산하고 내림차순 표 `vif_df`를 출력합니다. 기존 `X_train_scaled`를 사용해도 됩니다. 표준화는 특성 간 선형 관계 자체를 제거하지 않습니다.
2. 이번 과제에서는 `VIF < 5`: 상대적으로 낮음, `5 ≤ VIF < 10`: 주의, `VIF ≥ 10`: 높은 수준으로 분류합니다. 이는 절대적 삭제 규칙이 아닌 실습용 판단 기준입니다.
3. 주의·높은 수준인 특성을 특성명과 VIF로 제시합니다. 없으면 없다고 작성합니다. 완전한 선형 중복으로 VIF가 무한대인 경우 높은 수준으로 해석합니다. 상수 특성의 VIF는 정의하기 어려우므로 별도로 표시합니다.
4. 다중공선성이 계수의 크기·부호·안정성과 개별 변수 해석에 미치는 영향을 설명합니다.
5. 특성의 의미와 VIF 결과를 근거로 **변수 제거·결합·규제 중 한 가지 처치 방향**을 선택하고 대상·이유·한계를 설명합니다. 높은 VIF가 없으면 불필요한 제거를 하지 않는 이유와 기존 규제 비교의 목적을 설명해도 됩니다. 제거·결합의 추가 구현은 요구하지 않습니다.

### B. 기존 규제 모델 비교

다음 설정을 그대로 사용합니다.

```python
Ridge(alpha=1.0)
Lasso(alpha=1000, max_iter=10000)
ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
```

1. 세 모델을 동일한 표준화 Train 데이터로 학습합니다.
2. 동일한 Test 데이터의 R²를 계산합니다.
3. 전체 계수 크기 `abs(coef_).sum()`과 계수가 정확히 0인 개수를 계산합니다.
4. 결과를 `comparison_df`로 정리하고, 특성별 계수를 `task_coef_df`로 출력합니다.
5. Test R²와 계수 처리 차이를 함께 설명하고 Q5~Q9에 답합니다.

> Test 점수는 지정된 세 설정의 결과를 비교·해석하는 데 사용합니다. 이 점수를 보고 변수를 제거하거나 alpha를 반복 조정하지 않습니다. 실제 모델 선택·튜닝에는 Train 내부 검증 또는 CV가 필요합니다.

### 확인 포인트

- Train 입력 특성만 사용하여 VIF를 계산했는가?
- 기준과 실제 수치를 근거로 주의할 특성을 판단했는가?
- 다중공선성의 계수 해석 영향과 처치 선택 이유를 설명했는가?
- 규제가 입력 특성의 VIF 자체를 낮추는 것은 아님을 구분했는가?
- 같은 데이터에서 세 모델의 Test R²·계수 크기·0 계수 개수를 비교했는가?


In [ ]:
# TODO
# A. Train 특성별 VIF 계산 및 판단
# B. 기존 세 규제 모델 비교 및 특성별 계수 표 출력
# Q5~Q9 답변 작성

from sklearn.linear_model import LinearRegression

vif_list = []

for i, col in enumerate(features):
    target = X_train_scaled[:, i]                     
    others = np.delete(X_train_scaled, i, axis=1)     

    aux = LinearRegression()                          
    aux.fit(others, target)
    r2 = aux.score(others, target)

    vif = 1 / (1 - r2) 
    vif_list.append([col, r2, vif])

vif_df = pd.DataFrame(vif_list, columns=['feature', 'aux_r2', 'VIF'])
vif_df = vif_df.sort_values('VIF', ascending=False).reset_index(drop=True)

def vif_level(v):
    if v >= 10:
        return '높은 수준'
    elif v >= 5:
        return '주의'
    else:
        return '상대적으로 낮음'

vif_df['level'] = vif_df['VIF'].apply(vif_level)
display(vif_df.round(3))

print('상수 특성:', list(X_train.columns[X_train.std() == 0]))

print(vif_df[vif_df['VIF'] >= 5][['feature', 'VIF', 'level']].round(2))

models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1000, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
}

comp = []
task_coef_df = pd.DataFrame({'feature': features})

for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    r2 = r2_score(y_test, m.predict(X_test_scaled))
    comp.append([name, r2, abs(m.coef_).sum(), (m.coef_ == 0).sum()])
    task_coef_df[name] = m.coef_

comparison_df = pd.DataFrame(comp, columns=['model', 'test_r2', 'coef_sum', 'zero_count'])
display(comparison_df)
display(task_coef_df.round(1))

,feature,aux_r2,VIF,level
0,GrLivArea,0.992,128.777,높은 수준
1,2ndFlrSF,0.989,91.621,높은 수준
2,1stFlrSF,0.986,73.586,높은 수준
3,GarageCars,0.811,5.291,주의
4,GarageArea,0.801,5.032,주의
5,TotalBsmtSF,0.736,3.793,상대적으로 낮음
6,TotRmsAbvGrd,0.700,3.333,상대적으로 낮음
7,OverallQual,0.628,2.689,상대적으로 낮음
8,YearBuilt,0.581,2.389,상대적으로 낮음
9,FullBath,0.552,2.234,상대적으로 낮음


상수 특성: []
      feature     VIF  level
0   GrLivArea  128.78  높은 수준
1    2ndFlrSF   91.62  높은 수준
2    1stFlrSF   73.59  높은 수준
3  GarageCars    5.29     주의
4  GarageArea    5.03     주의


,model,test_r2,coef_sum,zero_count
0,Ridge,0.799331,102149.831747,0
1,Lasso,0.796018,88648.006135,2
2,ElasticNet,0.797291,99258.844252,0


,feature,Ridge,Lasso,ElasticNet
0,OverallQual,25310.9,25520.7,23514.6
1,GrLivArea,13960.7,21419.6,12670.8
2,GarageCars,7041.5,6820.2,6785.1
3,GarageArea,4383.0,4370.1,5137.4
4,TotalBsmtSF,7238.6,7305.6,8016.1
5,1stFlrSF,10011.0,3521.1,9237.8
6,2ndFlrSF,7098.9,0.0,6873.8
7,FullBath,-3154.8,-0.0,-1895.9
8,TotRmsAbvGrd,1602.1,83.2,2541.5
9,YearBuilt,8343.1,6966.9,8072.3


### Q5. Ridge와 Lasso의 가장 큰 계수 처리 차이는 무엇인가요?

**답변:** Ridge는 계수를 전체적으로 줄이기만 하고 0으로 만들지는 않아서 12개 계수가 다 남아 있었다. 반면 Lasso는 2ndFlrSF랑 FullBath 2개를 아예 0으로 만들었다.

### Q6. 규제 모델의 성능을 비교할 때 Test R²뿐 아니라 계수도 함께 보는 이유는 무엇인가요?

**답변:** 손실함수를 구할 때 R2 + 계수값에 대한 패널티 공식을 사용하기 때문에 계수도 함께 고려해 성능을 비교하는 것이다.

### Q7. 다중공선성은 회귀계수의 크기·부호·안정성과 개별 변수의 영향 해석에 어떤 문제를 일으킬 수 있나요?

**답변:** 특성들이 서로 비슷한 정보를 가지고 있으면, 모델이 어떤 특성 때문에 가격이 오르는지 구분을 잘 못한다. 그래서 같은 영향을 여러 특성이 나눠 갖게 되고, 모델마다 계수 크기가 크게 달라진다. 실제로 GrLivArea가 Ridge에선 13,961, Lasso에선 21,420이었다.
부호가 이상하게 나올 수도 있다. Ridge에서 FullBath가 -3,155로 나왔는데, 욕실이 많다고 집값이 떨어지는 건 말이 안 된다. 면적이나 방 개수랑 겹치는 정보라서 그런 것 같다.
또 데이터가 조금만 바뀌어도 계수가 크게 흔들릴 수 있다. 그래서 "이 특성이 1 늘면 가격이 얼마 오른다" 같은 해석을 그대로 믿기 어렵다. 다만 전체 예측 성능은 크게 안 나빠질 수도 있다.


### Q8. VIF 판단 기준은 무엇이며, 계산 결과에서 주의하거나 처치를 검토할 특성은 무엇인가요? 특성명과 VIF를 근거로 답하세요.

**답변:** 이번 과제 기준은 VIF 5 미만이면 낮음, 5 이상 10 미만이면 주의, 10 이상이면 높은 수준이다. 높은 수준은 GrLivArea, 2ndFlrSF, 1stFlrSF 세 개였다. GrLivArea가 생활 면적이라 거의 1층 면적 + 2층 면적이랑 같아서, 서로 거의 완전히 설명되는 것 같다. 주의는 GarageCars, GarageArea였다. 둘 다 차고 크기를 나타내는 특성이라 겹치는 것 같다. 나머지 7개는 다 5 미만이었고 상수 특성은 없었다.


### Q9. VIF 결과와 특성의 의미를 고려하여 변수 제거·결합·규제 중 어떤 처치 방법을 선택하겠나요? 구체적인 대상, 선택 이유와 한계를 설명하세요.

**답변:** 변수 제거를 선택하겠다. 1stFlrSF랑 2ndFlrSF를 빼고 GrLivArea만 남기는 방법이다.GrLivArea가 사실상 두 특성을 합친 값이라 정보가 거의 겹치고, VIF도 74~129로 엄청 높다. 세 개를 다 넣으면 계수가 나눠져서 해석이 어려운데, 하나만 남기면 면적 영향을 계수 하나로 볼 수 있다. Lasso도 2ndFlrSF를 0으로 만들고 GrLivArea에 몰아준 걸 보면 비슷한 판단인 것 같다. 차고 쪽은 5를 살짝 넘는 정도라서 이번엔 빼지 않았다.
한계도 존재한다. 두 특성을 빼면 1층이랑 2층 면적 비율, 그러니까 단층집인지 2층집인지 같은 정보가 사라진다. 그리고 VIF를 Train 데이터 하나로만 계산했고, 실제로 빼고 나서 성능이 어떻게 되는지는 이번에 확인 안 했다. 또 규제를 쓰면 계수가 흔들리는 건 줄일 수 있지만, 특성끼리 겹치는 관계 자체가 없어지는 건 아니다.

---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 규제는 손실함수에 무엇을 추가하는 방법인가요?
2. alpha가 커지면 규제 강도는 어떻게 변하나요?
3. Ridge의 계수는 일반적으로 어떻게 변하나요?
4. Lasso가 변수 선택 효과를 가지는 이유는 무엇인가요?
5. ElasticNet의 `l1_ratio=0.5`는 어떤 의미인가요?
6. 규제 전에 특성 스케일을 맞추는 이유는 무엇인가요?

- 기존 손실함수에 계수 크기에 대한 penalty를 추가한다.
- alpha가 커지면 규제가 강해져서 계수가 더 많이 줄어든다.
- 계수가 전체적으로 0에 가깝게 줄어들지만, 정확히 0이 되지는 않는다.
- L1 규제를 쓰기 때문에, 덜 중요한 특성의 계수를 정확히 0으로 만들 수 있어서다.
- 규제에서 L1과 L2를 반반 섞어 쓴다는 의미다.
- 규제는 계수 크기에 벌점을 주는데, 특성의 단위가 다르면 계수 크기도 단위 때문에 달라진다. 그러면 특성마다 벌점이 불공평하게 들어가서 스케일을 맞춰야 한다.